In [2]:
import pygame
import random
import math

pygame 2.6.1 (SDL 2.28.4, Python 3.12.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Brick Game Racing v1

In [5]:
pygame.init()

# Screen setup
WIDTH, HEIGHT = 500, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Mini Racer")

# Colors
WHITE = (255, 255, 255)
GRAY = (50, 50, 50)
RED = (200, 0, 0)

# Clock
clock = pygame.time.Clock()
FPS = 60

# Car setup
car_width, car_height = 50, 100
player_x = WIDTH // 2 - car_width // 2
player_y = HEIGHT - car_height - 20
player_speed = 5

# Enemy cars
enemy_width, enemy_height = 50, 100
enemy_speed = 5
enemy_list = []

def spawn_enemy():
    x = random.randint(0, WIDTH - enemy_width)
    y = random.randint(-150, -100)
    enemy_list.append(pygame.Rect(x, y, enemy_width, enemy_height))

# Game loop
running = True
while running:
    screen.fill(GRAY)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # Player movement
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT] and player_x > 0:
        player_x -= player_speed
    if keys[pygame.K_RIGHT] and player_x < WIDTH - car_width:
        player_x += player_speed

    player_rect = pygame.Rect(player_x, player_y, car_width, car_height)
    pygame.draw.rect(screen, WHITE, player_rect)

    # Spawn enemies
    if len(enemy_list) < 5:
        spawn_enemy()

    # Move enemies
    for enemy in enemy_list:
        enemy.y += enemy_speed
        pygame.draw.rect(screen, RED, enemy)

    # Collision detection
    for enemy in enemy_list:
        if player_rect.colliderect(enemy):
            print("Crash!")
            running = False

    # Remove off-screen enemies
    enemy_list = [e for e in enemy_list if e.y < HEIGHT]

    pygame.display.update()
    clock.tick(FPS)

pygame.quit()



Crash!


## Brick Game Racing v2
- Added Menu
- Added Retry/Quit

In [ ]:
import pygame
import random
import sys

In [8]:
pygame.init()

# Screen setup
WIDTH, HEIGHT = 500, 700
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Mini Racer")

# Colors
WHITE = (255, 255, 255)
GRAY = (50, 50, 50)
RED = (200, 0, 0)
BLACK = (0, 0, 0)

# Clock
clock = pygame.time.Clock()
FPS = 60

# Car setup
car_width, car_height = 50, 100
player_speed = 5

# Enemy cars
enemy_width, enemy_height = 50, 100
enemy_list = []

# Game state
state = "menu"  # "menu", "playing", "gameover"

# Difficulty settings
difficulties = {
    "Slow": {"enemy_speed": 3, "max_enemies": 3},
    "Moderate": {"enemy_speed": 5, "max_enemies": 5},
    "Fast": {"enemy_speed": 8, "max_enemies": 7}
}
current_difficulty = "Moderate"
enemy_speed = difficulties[current_difficulty]["enemy_speed"]
max_enemies = difficulties[current_difficulty]["max_enemies"]

# Fonts
font_large = pygame.font.SysFont(None, 60)
font_medium = pygame.font.SysFont(None, 40)

def draw_button(text, x, y, w, h, color, hover_color, action=None):
    mouse = pygame.mouse.get_pos()
    click = pygame.mouse.get_pressed()

    rect = pygame.Rect(x, y, w, h)
    if rect.collidepoint(mouse):
        pygame.draw.rect(screen, hover_color, rect)
        if click[0] == 1 and action is not None:
            action()
    else:
        pygame.draw.rect(screen, color, rect)

    label = font_medium.render(text, True, WHITE)
    screen.blit(label, (x + (w - label.get_width()) // 2, y + (h - label.get_height()) // 2))

def spawn_enemy():
    x = random.randint(0, WIDTH - enemy_width)
    y = random.randint(-150, -100)
    enemy_list.append(pygame.Rect(x, y, enemy_width, enemy_height))

def start_game():
    global state, enemy_list, player_x, player_y, enemy_speed, max_enemies
    state = "playing"
    enemy_list = []
    player_x = WIDTH // 2 - car_width // 2
    player_y = HEIGHT - car_height - 20
    enemy_speed = difficulties[current_difficulty]["enemy_speed"]
    max_enemies = difficulties[current_difficulty]["max_enemies"]

def change_difficulty():
    global current_difficulty
    options = list(difficulties.keys())
    idx = options.index(current_difficulty)
    current_difficulty = options[(idx + 1) % len(options)]

def quit_game():
    pygame.quit()
    sys.exit()

def menu_screen():
    screen.fill(GRAY)
    title = font_large.render("Mini Racer", True, WHITE)
    screen.blit(title, (WIDTH//2 - title.get_width()//2, 150))
    draw_button("Play", 180, 250, 150, 60, RED, (255,0,0), start_game)
    draw_button(f"Difficulty: {current_difficulty}", 100, 350, 300, 60, RED, (255,0,0), change_difficulty)
    draw_button("Quit", 180, 450, 150, 60, RED, (255,0,0), quit_game)
    pygame.display.update()

def gameover_screen():
    screen.fill(GRAY)
    label = font_large.render("Game Over!", True, WHITE)
    screen.blit(label, (WIDTH//2 - label.get_width()//2, 200))
    draw_button("Retry", 180, 300, 150, 60, RED, (255,0,0), start_game)
    draw_button("Quit", 180, 400, 150, 60, RED, (255,0,0), quit_game)
    pygame.display.update()

# Main loop
running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    if state == "menu":
        menu_screen()

    elif state == "playing":
        screen.fill(GRAY)

        # Player movement
        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            player_x -= player_speed
        if keys[pygame.K_RIGHT]:
            player_x += player_speed
        player_x = max(0, min(WIDTH - car_width, player_x))

        player_rect = pygame.Rect(player_x, player_y, car_width, car_height)
        pygame.draw.rect(screen, WHITE, player_rect)

        # Spawn enemies
        if len(enemy_list) < max_enemies:
            spawn_enemy()

        # Move enemies
        for enemy in enemy_list:
            enemy.y += enemy_speed
            pygame.draw.rect(screen, RED, enemy)

        # Collision detection
        for enemy in enemy_list:
            if player_rect.colliderect(enemy):
                state = "gameover"

        # Remove off-screen enemies
        enemy_list = [e for e in enemy_list if e.y < HEIGHT]

        pygame.display.update()

    elif state == "gameover":
        gameover_screen()

    clock.tick(FPS)

pygame.quit()